In [1]:
import cv2
import numpy as np
import os
import mediapipe as mp

# MediaPipe Setup
mp_holistic = mp.solutions.holistic 
mp_drawing = mp.solutions.drawing_utils 

def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image, results

def draw_styled_landmarks(image, results):
    # Face mesh (1404 points)
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_CONTOURS,
                             mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1),
                             mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1))
    # Pose (132 points)
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)) 
    # Left Hand (63 points)
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)) 
    # Right Hand (63 points)
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))

def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, lh, rh]) # Total: 1662

In [3]:
VIDEO_PATH = r"C:\Users\HP\Desktop\modeltraining2\input_video"
DATA_PATH = r"C:\Users\HP\Desktop\modeltraining2\output_video"

actions = np.array(['Beautiful', 'Drink', 'Eat', 'Five', 'Good', 'Hello', 'House', 'Love', 'Man', 'Mother', 'Run', 'Thank you', 'White', 'Yellow', 'You'])
no_videos = 68
sequence_length = 45

for action in actions:
    for video in range(no_videos):
        os.makedirs(os.path.join(DATA_PATH, action, str(video)), exist_ok=True)

In [5]:
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    for action in actions:
        print(f"--- Processing Action: {action} ---")
        for video in range(no_videos):
            # Check for file
            video_file = None
            for ext in ['.mp4', '.MP4', '.avi']:
                test_path = os.path.join(VIDEO_PATH, action, f"{video}{ext}")
                if os.path.exists(test_path):
                    video_file = test_path
                    break
            
            if not video_file:
                continue

            cap = cv2.VideoCapture(video_file)
            frame_num = 0
            
            while frame_num < sequence_length:
                ret, frame = cap.read()
                
                target_path = os.path.join(DATA_PATH, action, str(video), f"{frame_num}.npy")
                
                if not ret:
                    # VIDEO ENDED EARLY: Pad with zeros so training doesn't crash
                    keypoints = np.zeros(1662)
                    np.save(target_path, keypoints)
                else:
                    # Process Frame
                    image, results = mediapipe_detection(frame, holistic)
                    draw_styled_landmarks(image, results)
                    
                    # Save real data
                    keypoints = extract_keypoints(results)
                    np.save(target_path, keypoints)
                    
                    # Show progress
                    cv2.putText(image, f'Collecting {action} Video {video} Frame {frame_num}', (15,30), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2, cv2.LINE_AA)
                    cv2.imshow('OpenCV Feed', image)

                frame_num += 1
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            
            cap.release()
            print(f"      [OK] Video {video} complete.")

    cv2.destroyAllWindows()
print("DATA COLLECTION FINISHED! 15 SIGNS X 68 VIDEOS READY!")

--- Processing Action: Beautiful ---


      [OK] Video 0 complete.


      [OK] Video 1 complete.


      [OK] Video 2 complete.


      [OK] Video 3 complete.


...

      [OK] Video 63 complete.


      [OK] Video 64 complete.


      [OK] Video 65 complete.


      [OK] Video 66 complete.


      [OK] Video 67 complete.
DATA COLLECTION FINISHED! 15 SIGNS X 68 VIDEOS READY!
